<a href="https://colab.research.google.com/github/tousif47/Geostormcaster/blob/main/flare_parse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
flare_parse.py — GOES XRS flare report -> per-hour flare features CSV
Input : sci_xrsf-l2-flrpt_geo_s19950103_e20261027_v1-0-1.csv (GOES 8-19,
        1995-2026, science quality, one row per detected flare)
Output: flares.csv indexed by hour (UTC, tz-naive to match omni_clean.csv):
        n_24h, n_7d      flare counts in trailing 24h / 7d windows
        flux_24h, flux_7d  summed peak XRS-B irradiance in those windows
                           (W/m^2; log-scaled, energy-release proxy)
        max_24h, max_7d  strongest single flare (irradiance) in windows

Self-check at the end validates against the known ~11-year solar cycle:
flare counts must peak near 2001, 2014, and 2025-2026.
"""
import pandas as pd
import numpy as np

FLARE_CSV = "/content/drive/MyDrive/temp/sci_xrsf-l2-flrpt_geo_s19950103_e20261027_v1-0-1.csv"
OUT_CSV   = "/content/drive/MyDrive/temp/flares.csv"

# --- load and basic clean ---
fl = pd.read_csv(FLARE_CSV, parse_dates=["time", "start_time", "end_time"])
fl = fl.dropna(subset=["time", "flare_class"])
fl["time"] = pd.to_datetime(fl["time"], utc=True, errors="coerce").dt.tz_localize(None)
fl["xrsb_irrad"] = pd.to_numeric(fl["xrsb_irrad"], errors="coerce")

# class letter/magnitude kept for the sanity print only; xrsb_irrad carries
# the physical energy information used for features
fl["class_letter"] = fl["flare_class"].astype(str).str[0].str.upper()

print(f"flares loaded: {len(fl)}")
print(fl["class_letter"].value_counts().sort_index().to_string())

# --- aggregate flares into per-hour bins keyed by peak time ---
per_hour = fl.set_index(fl["time"].dt.floor("h"))["xrsb_irrad"] \
             .groupby(level=0).agg(["count", "sum", "max"])

# full hourly grid covering the flare era; reindex so empty hours = 0
idx = pd.date_range("1995-01-01 00:00", "2026-12-31 23:00", freq="h")
full = per_hour.reindex(idx).fillna(0.0)

# --- trailing-window features (window ends at the current hour, inclusive) ---
out = pd.DataFrame(index=idx)
out["n_24h"]  = full["count"].rolling(24).sum()
out["n_7d"]   = full["count"].rolling(24 * 7).sum()
out["flux_24h"] = np.log10(full["sum"].rolling(24).sum() + 1e-12)
out["flux_7d"]  = np.log10(full["sum"].rolling(24 * 7).sum() + 1e-12)
out["max_24h"] = np.log10(full["max"].rolling(24).max() + 1e-12)
out["max_7d"]  = np.log10(full["max"].rolling(24 * 7).max() + 1e-12)

# first 7 days have incomplete windows -> drop
out = out.iloc[24 * 7:]

out.to_csv(OUT_CSV)
print(f"\nSaved {OUT_CSV}: {len(out)} hours, "
      f"{out.index[0]} -> {out.index[-1]}")

# --- self-check: yearly flare counts must follow the solar cycle ---
yearly = full["count"].resample("YE").sum()
print("\n===== SELF-CHECK (yearly flare count vs solar cycle) =====")
print(yearly.round(0).astype(int).to_string())

/tmp/ipykernel_3258/686873688.py:21: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  fl = pd.read_csv(FLARE_CSV, parse_dates=["time", "start_time", "end_time"])


flares loaded: 81958
class_letter
A       15
B    29845
C    46038
M     5694
X      366

Saved /content/drive/MyDrive/temp/flares.csv: 280344 hours, 1995-01-08 00:00:00 -> 2026-12-31 23:00:00

===== SELF-CHECK (yearly flare count vs solar cycle) =====
1995-12-31    1742
1996-12-31     902
1997-12-31    2135
1998-12-31    3681
1999-12-31    3569
2000-12-31    3719
2001-12-31    3669
2002-12-31    3590
2003-12-31    3662
2004-12-31    3666
2005-12-31    3359
2006-12-31    2199
2007-12-31    1136
2008-12-31     137
2009-12-31     389
2010-12-31    2208
2011-12-31    3382
2012-12-31    3300
2013-12-31    3217
2014-12-31    3599
2015-12-31    3352
2016-12-31    2836
2017-12-31    2253
2018-12-31     597
2019-12-31     463
2020-12-31     584
2021-12-31    2197
2022-12-31    3333
2023-12-31    3363
2024-12-31    3869
2025-12-31    3280
2026-12-31    2570
Freq: YE-DEC
